# 09.正式模型开发报告

生成多 Sheet Excel 形式的正式模型开发报告。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
final_report_path = OUTPUT_DIR / f"8.最终模型结论整理_{customer_type}.xlsx"

final_all = pd.read_excel(final_report_path, sheet_name="1_全部模型结果")
final_rank = pd.read_excel(final_report_path, sheet_name="2_最终模型排序")
final_conclusion = pd.read_excel(final_report_path, sheet_name="3_最终推荐结论")

project_info = pd.DataFrame([
    ["客户类型", customer_type],
    ["样本规模", "demo sample size"],
    ["样本划分", "train / valid / oot"],
    ["建模标签", "y1 / y2"],
    ["建模目标", "构建风控风险区分模型，并比较 LGBM 与 XGB、含评分与不含评分效果"],
    ["评估指标", "AUC、KS、OOT baseline、train-valid-oot 衰减、决策表排序性"],
], columns=["项目", "说明"])

filter_process = pd.DataFrame([
    ["01_无监督筛选", "计算匹配率、同值率、PSI"],
    ["02_有监督筛选", "计算变量AUC、KS、分月KS"],
    ["03_单变量初筛", "应用PSI、KS、基础统计量规则"],
    ["04_决策树初筛", "使用树模型进行变量和产品筛选"],
    ["05_LGBM建模", "训练LGBM并输出模型效果和决策表"],
    ["07_XGB建模", "使用XGB进行算法对比和优化"],
    ["08_最终结论", "汇总全部模型并形成推荐"],
], columns=["步骤", "说明"])

decision_table_summary = pd.DataFrame([
    ["LGBM_y1_含评分4B", "等频10分箱", "无逆序", "满足风控决策表要求"],
    ["LGBM_y2_含评分4A", "等频10分箱", "无逆序", "满足风控决策表要求"],
    ["XGB_y2_含评分4A", "等频10分箱", "基本无逆序", "具备较好业务应用价值"],
], columns=["模型", "分箱方式", "排序性", "结论"])

score_effect_summary = pd.DataFrame([
    ["LGBM_y1", 0.265075, 0.294041, 0.028966],
    ["LGBM_y2", 0.168198, 0.226409, 0.058211],
    ["XGB_y1", 0.243054, 0.279228, 0.036174],
    ["XGB_y2", 0.165911, 0.279336, 0.113425],
], columns=["模型", "不含评分_OOT_KS", "含评分_OOT_KS", "KS提升"])

report_conclusion = pd.DataFrame([
    ["总体结论", "含评分模型整体优于不含评分模型，说明基础评分变量具有较强风险区分能力。"],
    ["算法对比", "XGB在y2含评分场景下OOT KS较高，但LGBM在y1含评分场景下表现最好。"],
    ["推荐方案", "综合OOT KS、baseline达标情况及模型稳定性，推荐优先考虑LGBM y1 含评分4B模型。"],
], columns=["结论项", "内容"])

In [ ]:
algo_compare = final_all.pivot_table(
    index=["建模标签", "模型版本"],
    columns="算法",
    values=["OOT_KS", "OOT_AUC"],
    aggfunc="max"
).reset_index()

algo_compare.columns = [
    "_".join([str(i) for i in col if str(i) != ""])
    if isinstance(col, tuple) else col
    for col in algo_compare.columns
]

compare_score = final_all.copy()
compare_score["是否含评分"] = compare_score["模型版本"].apply(
    lambda x: "含评分" if str(x).startswith("含评分") else "不含评分"
)

recommend_detail = pd.DataFrame([
    ["最终推荐算法", final_conclusion["最终推荐算法"].iloc[0]],
    ["最终推荐标签", final_conclusion["最终推荐标签"].iloc[0]],
    ["最终推荐模型版本", final_conclusion["最终推荐模型版本"].iloc[0]],
    ["最终OOT KS", final_conclusion["最终OOT_KS"].iloc[0]],
    ["最终OOT AUC", final_conclusion["最终OOT_AUC"].iloc[0]],
], columns=["项目", "内容"])

In [ ]:
report_path = OUTPUT_DIR / f"9.正式模型开发报告_{customer_type}.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    project_info.to_excel(writer, sheet_name="1_项目背景", index=False)
    filter_process.to_excel(writer, sheet_name="2_建模流程", index=False)
    final_all.to_excel(writer, sheet_name="3_全部模型结果", index=False)
    compare_score.to_excel(writer, sheet_name="4_含评分对比", index=False)
    algo_compare.to_excel(writer, sheet_name="5_算法对比", index=False)
    final_rank.to_excel(writer, sheet_name="6_最终模型排序", index=False)
    recommend_detail.to_excel(writer, sheet_name="7_最终推荐模型", index=False)
    decision_table_summary.to_excel(writer, sheet_name="8_决策表排序性", index=False)
    score_effect_summary.to_excel(writer, sheet_name="9_评分变量分析", index=False)
    report_conclusion.to_excel(writer, sheet_name="10_报告结论", index=False)

print("saved:", report_path)